Exercise 5: Interactive AI Prototyping with Gradio

In [2]:
%pip install gradio transformers torch accelerate

Note: you may need to restart the kernel to use updated packages.


Import libraries

In [3]:
import gradio as gr
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

Load the pretrained AI model

In [ ]:
# Check available device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# Lightweight Hugging Face model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

print("Model loaded successfully!")
     

Device: cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\NIC\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NIC\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Create the chatbot function

In [ ]:
def chatbot(message, history):

    # Prepare conversation history
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Give clear and concise answers."
        }
    ]

    # Include earlier conversation
    for item in history:
        if item["role"] in ["user", "assistant"] and isinstance(item["content"], str):
            messages.append({
                "role": item["role"],
                "content": item["content"]
            })

    # Add current user message
    messages.append({
        "role": "user",
        "content": message
    })

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    # Generate response
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

Create and launch Gradio chatbot interface

In [ ]:
# Close previously running Gradio interfaces
gr.close_all()

demo = gr.ChatInterface(
    fn=chatbot,

    title="AI Chatbot — Interactive AI Prototype",

    description=(
        "Welcome to the AI Chatbot! "
        "This interactive prototype demonstrates conversational "
        "Artificial Intelligence using Gradio and Hugging Face. "
        "Enter a question or select a sample prompt to begin."
    ),

    examples=[
        "What is Artificial Intelligence?",
        "Explain Machine Learning with an example.",
        "What is the role of NIC in Digital India?",
        "What is the difference between AI and Deep Learning?"
    ]
)

demo.launch(
    share=False,
    debug=True
)